# Feature Engineering Testing v3 — Evaluasi Model Final

Notebook ini memuat **konfigurasi v3 yang sudah dikunci** dan mengevaluasinya pada official CIFAR-10 test 10.000 gambar. Tidak ada GridSearchCV, perubahan bobot, atau pemilihan parameter dari label test. Bila model v3 tidak disimpan karena kalah pada development validation, Testing berhenti dengan pesan yang jelas. Hasil adalah evaluasi **post-exposure** karena test sudah pernah digunakan pada eksperimen sebelumnya.

## 1. Lokasi artefak dan pustaka

Gunakan direktori yang sama dengan notebook Training. File output v2 tidak diubah. Testing v3 hanya menulis hasil di `outputs/cifar10_7fitur_v3/fusion/`.

In [1]:
from pathlib import Path
import gc
import hashlib
import json
import os
import time

import joblib
import numpy as np
import tensorflow as tf
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler, Normalizer
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

SEED = 30092026
ROOT = Path.cwd().resolve()
if not (ROOT / "outputs" / "cifar10_7fitur_v2").is_dir() and (ROOT / "tugas-7" / "outputs" / "cifar10_7fitur_v2").is_dir():
    ROOT = ROOT / "tugas-7"
V2 = ROOT / "outputs" / "cifar10_7fitur_v2" / "fusion"
V3 = ROOT / "outputs" / "cifar10_7fitur_v3" / "fusion"
SELECTION_PATH = V3 / "laporan" / "selection.json"
MODEL_PATH = V3 / "model" / "fusion_v3_final.joblib"
BATCH_SIZE = 128
VARIANTS = ("rgb", "avg", "ntsc")
CLASS_NAMES = ("airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck")
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)
print("Testing v3, path:", V3)


I0000 00:00:1791498812.425561  229709 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791498812.465143  229709 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791498813.431471  229709 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Testing v3, path: /home/saga/anaconda3/envs/envGPU/BDA_Project/tugas-after-uts/outputs/cifar10_7fitur_v3/fusion


## 2. Definisi transformer identik dengan Training

Class ini diperlukan untuk memuat objek `GroupPreprocessor` dari file joblib setelah kernel di-restart. Tidak ada fitting scaler pada data test; hanya pemanggilan `transform`.

In [2]:
class GroupPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, boundaries, weights, block_l2=False):
        self.boundaries = boundaries
        self.weights = weights
        self.block_l2 = block_l2

    def fit(self, X, y=None):
        self.boundaries_ = tuple(int(i) for i in self.boundaries)
        if self.boundaries_[0] != 0 or self.boundaries_[-1] != X.shape[1]:
            raise ValueError("Batas fitur tidak sesuai")
        if len(self.weights) != len(self.boundaries_) - 1:
            raise ValueError("Bobot kelompok tidak sesuai")
        self.scalers_ = []
        for left, right in zip(self.boundaries_[:-1], self.boundaries_[1:]):
            self.scalers_.append(StandardScaler().fit(X[:, left:right]))
        return self

    def transform(self, X):
        if X.shape[1] != self.boundaries_[-1]:
            raise ValueError("Dimensi transform salah")
        groups = []
        for scaler, w, left, right in zip(self.scalers_, self.weights,
                                           self.boundaries_[:-1], self.boundaries_[1:]):
            block = scaler.transform(X[:, left:right]).astype(np.float32)
            if self.block_l2:
                block = Normalizer().transform(block)
            groups.append(block * np.float32(w))
        return Normalizer().transform(np.concatenate(groups, axis=1)).astype(np.float32)


## 3. Validasi selection, pipeline, dan provenance

Periksa hash seleksi dan archive v2 sebelum membuka official test. File model final harus membuktikan fitting menggunakan 50.000 sampel, tidak boleh sekadar model development 40K.

In [3]:
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1024*1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

if not SELECTION_PATH.is_file() or not MODEL_PATH.is_file():
    raise FileNotFoundError("Model final v3 belum ada. Training mungkin melewati final fit karena validation tidak melampaui v2; lihat development_report.json.")
selection = json.loads(SELECTION_PATH.read_text(encoding="utf-8"))
model = joblib.load(MODEL_PATH)
manifest_path = V2 / "laporan" / "combination_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
if (selection.get("run_id") != "cifar10_7fitur_v3" or
        model.get("run_id") != "cifar10_7fitur_v3" or
        model.get("selection_sha256") != sha256_file(SELECTION_PATH) or
        model.get("identity_sha256") != selection.get("identity_sha256") or
        model.get("fit_count") != 50000 or
        selection["source_manifest_sha256"] != sha256_file(manifest_path)):
    raise ValueError("Seleksi, model, atau provenance v3 tidak kompatibel")
if model["kind"] != selection["selected"]["kind"]:
    raise ValueError("Jenis model tidak sama dengan selection.json")
for rep, archive_sha in model["source_archives"].items():
    source = V2 / "fitur" / f"train_combined_{rep}.npz"
    if sha256_file(source) != archive_sha:
        raise ValueError(f"Arsip train {rep} berubah")
for variant, expected_hash in zip(VARIANTS, manifest["model_hashes"]):
    path = ROOT / "outputs" / "cifar10_7fitur_v1" / variant / "model" / "cnn.keras"
    if sha256_file(path) != expected_hash:
        raise ValueError(f"Checkpoint CNN {variant} berubah")
rep = (model["spec"]["rep"] if model["kind"] in ("rbf", "hybrid")
       else model["representation"])
groups = (model["spec"]["groups"] if model["kind"] in ("rbf", "hybrid")
          else "cnn3")
print("Artefak final tervalidasi:", model["kind"], "| representation", rep, "| groups", groups)


FileNotFoundError: Model final v3 belum ada. Training mungkin melewati final fit karena validation tidak melampaui v2; lihat development_report.json.

## 4. Muat official test setelah seleksi diverifikasi

Official test dipakai hanya untuk menghitung prediksi dan metrik dari model yang sudah dipilih. Tidak ada pengaturan ulang parameter menggunakan label test.

In [ ]:
from tensorflow.keras.datasets import cifar10
(_, _), (x_test, y_test) = cifar10.load_data()
y_test = y_test.ravel().astype(np.int64)
if x_test.shape != (10000,32,32,3) or len(y_test) != 10000:
    raise ValueError("Official CIFAR-10 test berbeda dari yang diharapkan")
print("Official test:", x_test.shape, y_test.shape)


## 5. Preprocessing gambar sesuai checkpoint

Rumus RGB, grayscale AVG, dan grayscale NTSC harus tetap sesuai dengan Combination v2. Transformasi horizontal flip dilakukan pada sumbu lebar gambar.

In [ ]:
def prepare_images(x, variant):
    x = np.asarray(x, dtype=np.float32) / 255.0
    if variant == "rgb":
        return x
    if variant == "avg":
        return np.mean(x, axis=-1, keepdims=True).astype(np.float32)
    if variant == "ntsc":
        return np.sum(x * np.array([0.299,0.587,0.114], np.float32),
                      axis=-1, keepdims=True).astype(np.float32)
    raise ValueError(f"Variant tidak dikenal: {variant}")


## 6. Ekstraksi CNN test dari model beku

Jika arsip test v2 cocok dengan representation terpilih, fitur CNN dan descriptor digunakan kembali untuk menghemat waktu. Jika berbeda, ekstraksi CNN dijalankan dengan `training=False` dan setiap CNN dimuat secara bergantian. Tidak ada proses fitting.

In [ ]:
def extract_test(images, variant, cnn_path, representation, batch_size=BATCH_SIZE):
    cnn = tf.keras.models.load_model(cnn_path, compile=False)
    extractor = tf.keras.Model(cnn.input, cnn.get_layer("embedding").output)
    if extractor.output.shape[-1] != 512:
        raise ValueError(f"Dimensi embedding {variant} salah")
    result = np.empty((len(images), 512), np.float32)
    for number, begin in enumerate(range(0, len(images), batch_size), 1):
        end = min(begin + batch_size, len(images))
        batch = prepare_images(images[begin:end], variant)
        value = extractor(batch, training=False).numpy()
        if representation == "B":
            value = .5*(value + extractor(np.ascontiguousarray(batch[:,:,::-1,:]), training=False).numpy())
        result[begin:end] = value
        if number % 15 == 0 or end == len(images):
            print(variant, end, "/", len(images), flush=True)
    del extractor, cnn
    tf.keras.backend.clear_session()
    gc.collect()
    if not np.isfinite(result).all():
        raise ValueError(f"Fitur {variant} tidak berhingga")
    return result

cached = None
v2_selection_path = V2 / "laporan" / "selection.json"
v2_test_path = V2 / "fitur" / "test_combined_final.npz"
if v2_selection_path.is_file() and v2_test_path.is_file():
    old_selection = json.loads(v2_selection_path.read_text(encoding="utf-8"))
    if old_selection.get("representation") == rep:
        with np.load(v2_test_path, allow_pickle=False) as a:
            if (str(a["selection_sha256"]) == sha256_file(v2_selection_path)
                and str(a["representation"]) == rep
                and np.array_equal(a["indices"], np.arange(10000))
                and np.array_equal(a["y"], y_test)
                and list(a["model_hashes"]) == manifest["model_hashes"]
                and a["features"].shape == (10000,2173)
                and np.isfinite(a["features"]).all()):
                cached = a["features"].astype(np.float32)
                print("Memakai cache test v2 yang cocok dengan representation", rep)

if cached is not None:
    cnn_test = {variant: cached[:,i*512:(i+1)*512] for i,variant in enumerate(VARIANTS)}
else:
    cnn_test = {}
    for variant in VARIANTS:
        cnn_path = ROOT / "outputs" / "cifar10_7fitur_v1" / variant / "model" / "cnn.keras"
        cnn_test[variant] = extract_test(x_test, variant, cnn_path, rep)
print("Fitur CNN siap:", {k: v.shape for k,v in cnn_test.items()})


## 7. Definisi handcrafted descriptor identik dengan v2

Fungsi HOG, LBP, HSV, dan RGB statistics berikut disalin dari notebook Testing v2 agar formula konsisten. Untuk model tiga CNN saja, descriptor tidak perlu dihitung.

In [ ]:
def handcrafted_descriptors(x, batch_size=128, verbose=1):
    # HOG: 9 orientasi unsigned, cell 8x8, block 2x2, L2 per block.
    # LBP: 8 tetangga radius 1, histogram 256 kode.
    # HSV: 16 bin per kanal. Statistik RGB: mean, std, skewness per kanal.
    import matplotlib.colors as mcolors
    outputs = {k: [] for k in ("hog", "lbp", "hsv", "rgb_stats")}
    total_batches = int(np.ceil(len(x) / batch_size))
    progress_interval = max(1, total_batches // 20)
    for begin in range(0, len(x), batch_size):
        rgb = np.asarray(x[begin:begin+batch_size], dtype=np.float32) / 255.0
        gray = np.sum(rgb * np.array([0.299, 0.587, 0.114], np.float32), axis=-1)
        gy, gx = np.gradient(gray, axis=(1, 2))
        magnitude = np.sqrt(gx*gx + gy*gy)
        orientation = np.mod(np.arctan2(gy, gx) * (180.0 / np.pi), 180.0)
        bins = np.minimum((orientation / 20.0).astype(np.int32), 8)
        cell_hist = np.zeros((len(rgb), 4, 4, 9), dtype=np.float32)
        for row in range(4):
            for col in range(4):
                b = bins[:, row*8:(row+1)*8, col*8:(col+1)*8]
                m = magnitude[:, row*8:(row+1)*8, col*8:(col+1)*8]
                for angle in range(9):
                    cell_hist[:, row, col, angle] = np.sum(m * (b == angle), axis=(1, 2))
        blocks = []
        for row in range(3):
            for col in range(3):
                block = cell_hist[:, row:row+2, col:col+2, :].reshape(len(rgb), -1)
                denominator = np.maximum(
                    np.linalg.norm(block, axis=1, keepdims=True), 1e-8
                )
                blocks.append(block / denominator)
        outputs["hog"].append(np.concatenate(blocks, axis=1))
        padded = np.pad(gray, ((0, 0), (1, 1), (1, 1)), mode="edge")
        lbp = np.zeros(gray.shape, dtype=np.uint8)
        offsets = [(-1,-1), (-1,0), (-1,1), (0,1), (1,1), (1,0), (1,-1), (0,-1)]
        for bit, (dy, dx) in enumerate(offsets):
            neighbor = padded[:, 1+dy:33+dy, 1+dx:33+dx]
            lbp |= (neighbor >= gray).astype(np.uint8) << bit
        hist = np.zeros((len(rgb), 256), dtype=np.float32)
        np.add.at(hist, (np.repeat(np.arange(len(rgb)), 1024), lbp.reshape(-1)), 1)
        outputs["lbp"].append(hist / 1024.0)
        hsv = mcolors.rgb_to_hsv(rgb)
        hsv_hist = np.zeros((len(rgb), 3, 16), dtype=np.float32)
        for channel in range(3):
            hsv_bin = np.minimum((hsv[..., channel] * 16).astype(np.int32), 15)
            np.add.at(hsv_hist[:, channel, :],
                      (np.repeat(np.arange(len(rgb)), 1024), hsv_bin.reshape(-1)), 1)
        outputs["hsv"].append(hsv_hist.reshape(len(rgb), -1) / 1024.0)
        flat = rgb.reshape(len(rgb), -1, 3)
        mean = flat.mean(axis=1)
        std = flat.std(axis=1)
        skew = np.mean(((flat - mean[:, None, :]) / np.maximum(std[:, None, :], 1e-6)) ** 3, axis=1)
        outputs["rgb_stats"].append(np.concatenate((mean, std, skew), axis=1))
        batch_number = begin // batch_size + 1
        if verbose == 1 and (
            batch_number % progress_interval == 0
            or batch_number == total_batches
        ):
            processed = min(begin + batch_size, len(x))
            print(f"Descriptor: {processed}/{len(x)} gambar", flush=True)
    result = {key: np.concatenate(parts).astype(np.float32) for key, parts in outputs.items()}
    assert [result[k].shape[1] for k in ("hog", "lbp", "hsv", "rgb_stats")] == [324, 256, 48, 9]
    assert all(np.isfinite(value).all() for value in result.values())
    return result


## 8. Siapkan matriks fitur final

Pada `cnn3` digunakan 1.536 fitur CNN. Pada `all7`, descriptor diambil dari cache v2 yang tervalidasi atau dihitung dengan rumus yang sama seperti sebelumnya. Susunan tujuh kelompok tidak boleh berubah.

In [ ]:
cnn_matrix = np.concatenate([cnn_test[v] for v in VARIANTS], axis=1).astype(np.float32)
if groups == "cnn3":
    X_test_final = cnn_matrix
else:
    if cached is not None:
        descriptor_matrix = cached[:, 1536:]
        print("Descriptor test menggunakan cache v2")
    else:
        desc = handcrafted_descriptors(x_test, batch_size=BATCH_SIZE, verbose=1)
        descriptor_matrix = np.concatenate([desc[k] for k in ("hog","lbp","hsv","rgb_stats")], axis=1)
    if descriptor_matrix.shape != (10000,637):
        raise ValueError("Dimensi descriptor test tidak sesuai")
    X_test_final = np.concatenate([cnn_matrix, descriptor_matrix], axis=1).astype(np.float32)
if X_test_final.shape != (10000, (1536 if groups=="cnn3" else 2173)) or not np.isfinite(X_test_final).all():
    raise ValueError("Matriks test tidak valid")
print("X_test v3:", X_test_final.shape)


## 9. Fungsi prediksi RBF, ensemble, dan late fusion

Setiap estimator memakai scaler hasil fitting 50K yang disimpan. Untuk ensemble, decision scores ditransformasikan menggunakan statistik training yang dibekukan. Perubahan bobot berdasarkan test tidak diperbolehkan.

In [ ]:
def score_transform(scores, stats):
    return (np.asarray(scores, np.float64) - stats["mean"]) / stats["std"]

def predict_frozen(model, X):
    kind = model["kind"]
    if kind == "rbf":
        Z = model["preprocessor"].transform(X)
        return model["rbf"].predict(Z)
    if kind == "hybrid":
        Z = model["preprocessor"].transform(X)
        rbf_s = score_transform(model["rbf"].decision_function(Z), model["rbf_stats"])
        linear_s = score_transform(model["linear"].decision_function(Z), model["linear_stats"])
        return np.argmax(rbf_s + model["linear_weight"]*linear_s, axis=1)
    if kind == "late":
        scores = []
        for i, domain in enumerate(model["domains"]):
            raw_scores = domain["pipeline"].decision_function(X[:,i*512:(i+1)*512])
            scores.append(score_transform(raw_scores, domain["stats"]))
        return np.argmax(sum(w*s for w,s in zip(model["weights"],scores)), axis=1)
    raise ValueError(f"Jenis model tidak dikenal: {kind}")

pred = predict_frozen(model, X_test_final)
if pred.shape != y_test.shape or not np.isin(pred, np.arange(10)).all():
    raise ValueError("Prediksi tidak valid")
print("Prediksi official test selesai:", len(pred), "sampel")


## 10. Classification report dan target minimum 95%

Syarat tugas adalah **minimal 95,00%**, yakni sedikitnya 9.500 prediksi benar dari 10.000 gambar. Tidak ada klaim target tercapai jika skor lebih rendah. Semua angka dibaca dari prediksi model yang dibekukan.

In [ ]:
correct = int(np.sum(pred == y_test))
accuracy = float(accuracy_score(y_test, pred))
macro_f1 = float(f1_score(y_test, pred, average="macro"))
print(classification_report(y_test, pred, target_names=CLASS_NAMES))
print(f"Official test: {correct}/10000 = {accuracy:.2%} | Macro-F1: {macro_f1:.2%}")
print("TARGET:", "TERCAPAI" if correct >= 9500 else "BELUM TERCAPAI")
print("Catatan: post-exposure testing; official test sudah digunakan pada eksperimen terdahulu.")


## 11. Simpan hasil v3 tanpa menimpa v2

Simpan metrik dan prediksi secara terpisah. Jika file v3 lama memakai hash model berbeda, hentikan untuk menghindari pencampuran percobaan. Confusion matrix dihitung dari prediksi yang sama.

In [ ]:
report = {"run_id": "cifar10_7fitur_v3", "kind": model["kind"],
          "representation": rep, "groups": groups,
          "selection_sha256": sha256_file(SELECTION_PATH),
          "model_sha256": sha256_file(MODEL_PATH),
          "n_test": int(len(y_test)), "correct": correct, "accuracy": accuracy,
          "macro_f1": macro_f1, "target_minimum": .95,
          "target_reached": correct >= 9500,
          "post_exposure_testing": True,
          "classification_report": classification_report(y_test, pred,
                target_names=CLASS_NAMES, output_dict=True),
          "confusion_matrix": confusion_matrix(y_test,pred).tolist()}
metrics_path = V3 / "laporan" / "test_metrics.json"
if metrics_path.exists():
    previous = json.loads(metrics_path.read_text(encoding="utf-8"))
    if (previous.get("selection_sha256") != report["selection_sha256"] or
            previous.get("model_sha256") != report["model_sha256"]):
        raise ValueError("Hasil test lama dari model berbeda; jangan timpa")
tmp = metrics_path.with_suffix(".json.tmp")
tmp.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
os.replace(tmp, metrics_path)
np.savez_compressed(V3 / "laporan" / "test_predictions.npz",
                    indices=np.arange(10000), y_true=y_test, y_pred=pred,
                    selection_sha256=report["selection_sha256"])
print("Hasil test v3:", metrics_path)


## 12. Visualisasi confusion matrix

Visualisasi disimpan hanya untuk laporan; tidak memengaruhi pemilihan hyperparameter atau hasil prediksi.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=CLASS_NAMES,
                                      xticks_rotation=45, ax=ax, colorbar=False)
ax.set_title(f"CIFAR-10 Feature Engineering v3 — {accuracy:.2%} (post-exposure)")
fig.tight_layout()
fig.savefig(V3 / "laporan" / "confusion_matrix.png", dpi=150)
plt.show()
